<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-14_March-17-2026/Lecture-14_PCA_tSNE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 14 - Principal Component Analysis (PCA) and t-SNE


Install RDKit and [LightGBM](https://lightgbm.readthedocs.io/en/stable/)

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit mols2grid

Import all basic pacakges

In [ ]:
# basic
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# RDkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

# For progress bar
from tqdm.auto import tqdm


Download dataset

In [ ]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/refs/heads/main/Assignment-2/"
dataset_filename="BradleyDoublePlusGoodMeltingPointDataset.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

This is to add progress bars panda task

In [ ]:
tqdm.pandas()

Read dataset

In [ ]:
data_mp = pd.read_csv("BradleyDoublePlusGoodMeltingPointDataset.csv")



In [ ]:
data_mp

In [ ]:
# Simplify by removing columns from the data frame
data_mp = data_mp.drop(columns=['csid','link','source','count','min','max','range'])

Let's visualize the molecules using mols2grid.

In [ ]:
property_names = list(rdMolDescriptors.Properties.GetAvailableProperties())
property_getter = rdMolDescriptors.Properties(property_names)

In [ ]:
def smi2props(smi):
    mol = Chem.MolFromSmiles(smi)
    props = None
    if mol:
        Chem.DeleteSubstructs(mol, Chem.MolFromSmarts("[#1X0]"))
        props = np.array(property_getter.ComputeProperties(mol))
    return props

In [ ]:
# These two ways are equilivant

data_mp['props'] = data_mp.smiles.progress_apply(smi2props)

# data_mp['props'] = data_mp['smiles'].progress_apply(smi2props)

In [ ]:
data_mp = data_mp.dropna(subset=['props'])

In [ ]:
data_mp[property_names] = data_mp['props'].tolist()

In [ ]:
features = data_mp[property_names]

In [ ]:
features

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Apply PCA to 2D features
pca = PCA()
pca.fit(scaled_features)

print("Explained variance ratio for 2D features:")
display(pca.explained_variance_ratio_)
print("Cumulative explained variance for 2D features:")
display(pca.explained_variance_ratio_.cumsum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the first two principal components for 2D features
pc = pca.transform(scaled_features)[:, :2]

plt.scatter(x=pc[:,0], y=pc[:,1], c=data_mp['mpC'], alpha=0.5)
plt.title('First Two Principal Components of 2D Features (colored by mpC)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(label='Value')
plt.show()



In [ ]:
from sklearn.manifold import TSNE

In [ ]:

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Apply PCA to 2D features
tsne = TSNE()
emb = tsne.fit_transform(scaled_features)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.scatter(x=emb[:,0], y=emb[:,1], c=data_mp['mpC'], alpha=0.5)
plt.title('First Two Principal Components of 2D Features (colored by mpC)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(label='Value')
plt.show()